In [1]:
# chain.get_graph().print_ascii()

## Sequential Chain


In [2]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# model define
model1 = ChatGroq(model="llama-3.1-8b-instant")
model2 = ChatOpenAI(model = "gpt-5.4-nano")

# prompt define
prompt1 = PromptTemplate(template= "Generate a detailed report on {topic}", input_variables=['topic'])
prompt2 = PromptTemplate(template="generate a 5 pointer summary from the given \n {text}", input_variables=['text'])

#output parser define
parser = StrOutputParser()

# chain create
chain = prompt1 | model1 | parser | prompt2 | model1 | parser

#getting ai response
result = chain.invoke({"topic":"Quantum machine learning"})
print(result)

Here's a 5-pointer summary of the report "Quantum Machine Learning: A Comprehensive Report":

1. **Introduction to Quantum Machine Learning**: Quantum machine learning (QML) is a rapidly evolving field that combines the principles of quantum mechanics and machine learning to develop new algorithms and models for solving complex problems.

2. **Key Components and Types of Quantum Machine Learning**: QML relies on quantum computing, machine learning, and quantum algorithms to solve machine learning problems. It includes types such as Quantum K-Means, Quantum Support Vector Machines (QSVMs), Quantum Generative Adversarial Networks (QGANs), and Quantum Neural Networks (QNNs).

3. **Advantages of Quantum Machine Learning**: QML has the potential to significantly improve the performance and efficiency of traditional machine learning algorithms, enabling the processing of vast amounts of data and tackling problems that are currently unsolvable. It offers speedup, scalability, and improved acc

In [4]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

## parallel chain

In [5]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

In [ ]:
from langchain_core.runnables import RunnableParallel

prompt1 = PromptTemplate(template = "Generate a sort and simle note from the given text : {text}", input_variables= ['text'])
prompt2 = PromptTemplate(template = "Generate t short question answers from the following text: {text}",input_variables= ['text'])


parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'note': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

prompt3 = PromptTemplate(template = "Merge the provided notes and quiz into a single document\n note = {note}, quiz = {quiz}", input_variables=['note','quiz'])
merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

response = chain.invoke({"text": text})
print(response)


**Support Vector Machines (SVMs)**

**Overview**
-----------

Support Vector Machines (SVMs) are a set of supervised learning methods used for classification, regression, and outliers detection.

**Advantages**
-------------

* Effective in high-dimensional spaces
* Can handle cases with more dimensions than samples
* Memory efficient
* Versatile with various kernel functions available
* Can specify custom kernels

**Disadvantages**
----------------

* Choosing the correct kernel function and regularization term is crucial to avoid over-fitting
* Does not provide direct probability estimates (requires additional calculation)
* Requires careful handling of sparse data for optimal performance

**Quiz: Support Vector Machines (SVMs)**
--------------------------------------

1. **What are Support Vector Machines (SVMs) used for?**  
    They are supervised learning methods for classification, regression, and outlier detection.

2. **Name one advantage of SVMs in high-dimensional spaces.** 

In [8]:
chain.get_graph().print_ascii()

           +--------------------------+            
           | Parallel<note,quiz>Input |            
           +--------------------------+            
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +------------+   
    | ChatGroq |                  | ChatOpenAI |   
    +----------+                  +------------+   
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

## Conditional Chain

In [19]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal


class Feedback(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description= "Give the sentiment of the feedback")

parser = PydanticOutputParser(pydantic_object= Feedback)
parser2 = StrOutputParser()

prompt1 = PromptTemplate(template="Classify the sentiment of the following feedback text into postive or negative \n{feedback} \n{format_instruction}",
                         input_variables= ['feedback'],
                         partial_variables= {'format_instruction': parser.get_format_instructions()}
                         )

classifier = prompt1 | model1 | parser


# prompt for positive feedback
prompt2 = PromptTemplate(template= "Write an appropriate response to this positive feedback \n{feedback}", input_variables=['feedback'])
# prompt for Negative feedback
prompt3 = PromptTemplate(template= "Write an appropriate response to this negative feedback \n{feedback}", input_variables=['feedback'])

branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'Positive', prompt2 | model1 | parser2),
    (lambda x: x.sentiment == 'Negative', prompt3 | model1 | parser2),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier | branch_chain

response = chain.invoke({'feedback':"I don't like this phone features and design . It looks very cheap"})
print(response)

Here's a possible response to negative feedback:

"I apologize that our product/service didn't meet your expectations. Can you please provide more details about your experience, including what specifically didn't meet your needs? This will help us better understand the issue and take steps to improve our product/service. We value your feedback and appreciate your time in sharing your concerns with us."

This response acknowledges the customer's negative sentiment, expresses apology for not meeting their expectations, and opens the door for further discussion to gather more information and potentially resolve the issue.
